# Pipeline benchmark — real end-to-end timing on CPU

Runs the *real* pipeline (no `TestContainer`, no `MockLlm`): real MarkItDown/EasyOCR extraction, the real Phi-4-mini-instruct GGUF model for node2/node3's SLM calls, and the real `all-MiniLM-L6-v2` embedding model for node4 — against a real sample PDF from `playground/samples/`.

Unlike `pipeline_end_to_end.ipynb` (which deliberately mocks the SLM and stubs extraction for speed/determinism), this notebook exists specifically to answer *how long does this actually take on CPU* — the number the OCR/embedding/SLM GPU-vs-CPU tradeoff discussion needs.

**Requires**: `src/classiflow/ingesta/models/Phi-4-mini-instruct-Q4_K_M.gguf` present on disk (~2.5 GB, not committed — see `Settings.NODE2_MODEL_PATH`). First run also downloads the embedding model from Hugging Face (one-time, then cached).

> **Kernel**: select the project's `.venv` kernel in the top-right picker.

## 1 — Imports and a dedicated benchmark database

In [1]:
import asyncio
import time
from pathlib import Path

import classiflow
from classiflow.database.base import Base
from classiflow.database.repositories.audit import SqlAuditRepository
from classiflow.database.repositories.hash import SqlHashRepository
from classiflow.events.broadcaster import EventBroadcaster
from classiflow.ingesta.coordinator import build_coordinator
from classiflow.ingesta.domain.state import JobState
from classiflow.ingesta.extract import TextExtractor
from classiflow.ingesta.extractors import MarkItDownExtractor, OCRExtractor
from classiflow.ingesta.llm_provider import get_llm_langchain
from classiflow.ingesta.nodes.node1_file_reception import FileReceptionNode
from classiflow.ingesta.nodes.node2_format_validation import FormatValidationNode
from classiflow.ingesta.nodes.node3_content_validation import ContentValidationNode
from classiflow.ingesta.nodes.node4_duplicate_control import DuplicateControlNode
from classiflow.services.audit.service import AuditService
from classiflow.settings import Settings
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

# Dedicated DB file (gitignored, see .gitignore's `data/**` / `*.db` rules) so repeated
# benchmark runs don't accumulate job/audit rows in the real data/classiflow.db.
_project_root = Path(classiflow.__file__).parents[2]
_db_path = _project_root / "data" / "pipeline_benchmark.db"
DB_URL = f"sqlite+aiosqlite:///{_db_path.as_posix()}"

engine = create_async_engine(DB_URL, echo=False)
session_factory = async_sessionmaker(engine, expire_on_commit=False)

async def _create_tables() -> None:
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)

await _create_tables()
print(f"DB ready at {_db_path}")

c:\Users\leona\source\repos\Trabajo-Integrador\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DB ready at C:\Users\leona\source\repos\Trabajo-Integrador\data\pipeline_benchmark.db


## 2 — A real sample document

In [2]:
_SAMPLES_DIR = Path(classiflow.__file__).parent / "playground" / "samples"
_FILENAME = "convenio_2_2013.pdf"
pdf_bytes = (_SAMPLES_DIR / _FILENAME).read_bytes()
print(f"{_FILENAME}: {len(pdf_bytes):,} bytes")

convenio_2_2013.pdf: 121,098 bytes


## 3 — Extraction timing (MarkItDown → EasyOCR fallback)

Timed standalone since the coordinator's `extract` step isn't itself audit-instrumented (only the 4 `BaseNode` subclasses are — see `ingesta/nodes/base.py`).

In [3]:
import easyocr

ocr_reader = easyocr.Reader([Settings.ocr_lang], gpu=True)
text_extractor = TextExtractor([MarkItDownExtractor(), OCRExtractor(reader=ocr_reader)])

t0 = time.perf_counter()
extracted_text = text_extractor(pdf_bytes, _FILENAME)
t_extract = time.perf_counter() - t0
print(f"extraction: {t_extract:.3f}s  ({len(extracted_text)} chars)")

extraction: 0.539s  (6814 chars)


## 4 — Isolated SLM timing: cold load vs. warm inference

Run *before* the full pipeline below, so this first call is genuinely cold (`get_llm_langchain` is `@lru_cache`d — once warm here, node3's call in the full pipeline run will reuse the same resident model).

In [4]:
_PROMPT = (
    "Respond with strict JSON only: "
    '{"is_legitimate": true, "confidence": 0.9, "reasoning": "short reason"}'
)

t0 = time.perf_counter()
llm = get_llm_langchain(Settings.node2_model_path)  # loads weights from disk
_ = llm.invoke(_PROMPT)
t_cold = time.perf_counter() - t0
print(f"SLM cold (load + first inference): {t_cold:.3f}s")

t0 = time.perf_counter()
_ = llm.invoke(_PROMPT)
t_warm = time.perf_counter() - t0
print(f"SLM warm (pure inference):          {t_warm:.3f}s")

llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


SLM cold (load + first inference): 26.147s
SLM warm (pure inference):          4.414s


## 5 — Full pipeline run (node1 → extract → node2 → node3 → node4)

SLM is already warm from section 4, so node3's duration below reflects steady-state per-document latency, not one-time model-load cost.

In [5]:
async def run_pipeline() -> dict:
    async with session_factory() as session:
        audit = AuditService(SqlAuditRepository(session))
        broadcaster = EventBroadcaster()
        node1 = FileReceptionNode(audit=audit, broadcaster=broadcaster)
        node2 = FormatValidationNode(audit=audit, broadcaster=broadcaster)
        node3 = ContentValidationNode(audit=audit, broadcaster=broadcaster)
        node4 = DuplicateControlNode(
            audit=audit, broadcaster=broadcaster, hash_repo=SqlHashRepository(session)
        )
        coordinator = build_coordinator(
            node1, node2, node3, node4, text_extractor=text_extractor
        )

        state: JobState = {"job_id": "bench-1", "filename": _FILENAME, "file_bytes": pdf_bytes}
        t0 = time.perf_counter()
        result = await coordinator.ainvoke(state)
        t_full = time.perf_counter() - t0
        await session.commit()
    return {"result": result, "t_full": t_full}

_run = await run_pipeline()
print(f"full pipeline: {_run['t_full']:.3f}s -> final_status={_run['result'].get('final_status')!r}")

2026-08-12 17:23:28.256 | INFO     | classiflow.services.audit.service:record:37 - audit | job=bench-1 node=node1_file_reception event=passed
2026-08-12 17:23:28.262 | INFO     | classiflow.services.audit.service:record:37 - audit | job=bench-1 node=node2_format_validation event=passed
2026-08-12 17:23:40.113 | INFO     | classiflow.services.audit.service:record:37 - audit | job=bench-1 node=node3_content_validation event=passed
2026-08-12 17:23:40.119 | INFO     | classiflow.services.audit.service:record:37 - audit | job=bench-1 node=node4_duplicate_control event=failed


full pipeline: 11.888s -> final_status='rejected'


## 6 — Which components actually ran on GPU

OCR and embeddings auto-detect a device (`torch.cuda.is_available()` under the hood); the SLM does not auto-detect anything — `n_gpu_layers` defaults to `0` in `llm_provider.py`, and even if it were set, `llama_cpp.llama_supports_gpu_offload()` reports whether the *installed build* has CUDA compiled in at all. Printing all three here so the timings above aren't misread as apples-to-apples.

In [6]:
import llama_cpp
import torch

import classiflow.ingesta.nodes.node4_duplicate_control as node4_module

_gpu_name = f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""
print(f"GPU visible to torch        : {torch.cuda.is_available()}{_gpu_name}")

# _get_sentence_model() is @lru_cache'd -- node4's run above already populated it, so
# this is a free cache hit, not a second model load.
embed_model = node4_module._get_sentence_model()
print(f"OCR reader device           : {ocr_reader.device}")
print(f"Embedding model device      : {embed_model.device}")

_gpu_offload_supported = llama_cpp.llama_supports_gpu_offload()
_n_gpu_layers = llm.client.model_params.n_gpu_layers
_slm_device = "GPU" if _gpu_offload_supported and _n_gpu_layers != 0 else "CPU"
print(f"SLM (llama-cpp-python)      : {_slm_device}  (n_gpu_layers={_n_gpu_layers}, installed build supports GPU offload={_gpu_offload_supported})")

GPU visible to torch        : True (NVIDIA RTX A4000 Laptop GPU)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6285.66it/s]


OCR reader device           : cuda
Embedding model device      : cuda:0
SLM (llama-cpp-python)      : CPU  (n_gpu_layers=0, installed build supports GPU offload=False)


## 7 — Per-node breakdown from the audit log

Every `BaseNode` subclass already records `duration_ms` per node (`ingesta/nodes/base.py:_emit_and_audit`) — no extra instrumentation needed, just read it back.

In [7]:
async def print_audit(job_id: str) -> None:
    async with session_factory() as session:
        repo = SqlAuditRepository(session)
        for record in await repo.list_for_job(job_id):
            print(f"  {record.node:<28} {record.event:<8} {record.duration_ms:>6} ms")

print("=== per-node durations (bench-1) ===")
await print_audit("bench-1")

=== per-node durations (bench-1) ===
  node1_file_reception         passed        0 ms
  node2_format_validation      passed        0 ms
  node3_content_validation     passed    10483 ms
  node4_duplicate_control      passed     4296 ms
  node1_file_reception         passed        0 ms
  node2_format_validation      passed        0 ms
  node3_content_validation     passed    11328 ms
  node4_duplicate_control      failed       15 ms


## 8 — Cleanup

In [8]:
await engine.dispose()
print("engine disposed")

engine disposed
